In [1]:
#%pip install "mistralai==1.8.1" pandas tqdm python-dotenv --quiet

In [ ]:
import os
import json
import time
from typing import Any
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from mistralai import Mistral

In [3]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
load_dotenv()

api_key = os.getenv("MISTRAL_API_KEY")
if not api_key:
    raise ValueError("❌ ERREUR : La clé MISTRAL_API_KEY n'est pas trouvée. As-tu créé le fichier .env ?") 
client = Mistral(api_key=api_key)

MODEL_NAME = "open-mistral-nemo" # Modèle Mistral rapide, moderne et économique

FICHIER_ENTREE = "D:/XoNet/data/base.csv" 
FICHIER_SORTIE = "D:/XoNet/data/corpus_final.csv"
COLONNE_TEXTE = "fr"
TAILLE_LOT = 150 # Lot size

# ==========================================
# 2. PROMPT SPÉCIFIQUE À NOTRE PROJET
# ==========================================
PROMPT_SYSTEME = """
Tu es un expert en analyse de sentiments en français.
On va te donner une liste de textes numérotés sous la forme :
index: texte

Tu dois classer chaque texte selon STRICTEMENT cette échelle :
- 0 : Neutre (fait brut, description, phrase biblique sans émotion)
- 1 : Négatif (tristesse, colère, refus, douleur, insulte, malédiction)
- 2 : Positif (joie, accord, beauté, bénédiction, bien-être)

Tu DOIS répondre exclusivement sous la forme d'un objet JSON contenant une liste "resultats" d'objets avec "index" et "sentiment".
Exemple de format attendu :
{
  "resultats": [
    {"index": 0, "sentiment": 0},
    {"index": 1, "sentiment": 2},
    {"index": 2, "sentiment": 1}
  ]
}
"""

In [ ]:
# ==========================================
# 3. FONCTION D'APPEL AVEC RETRY
# ==========================================
def appeler_api_avec_retry(user_prompt: str, max_retries: int = 5) -> dict[str, Any]:
    delay = 2.0
    for attempt in range(max_retries):
        try:
            response = client.chat.complete(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": PROMPT_SYSTEME},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.0
            )
            raw_content = response.choices[0].message.content
            content_text = str(raw_content or "")
            if not content_text.strip():
                raise ValueError("Réponse Mistral invalide: message.content vide")
            parsed = json.loads(content_text)
            if not isinstance(parsed, dict):
                raise ValueError("Réponse JSON invalide: objet attendu")
            return parsed
        except Exception as e:
            error_str = str(e)
            print(f"\n⚠️ [Tentative {attempt+1}/{max_retries}] Erreur : {error_str[:120]}")
            if attempt < max_retries - 1:
                sleep_time = delay * (1.5 ** attempt)
                print(f"Attente de {sleep_time:.1f}s avant de réessayer...")
                time.sleep(sleep_time)
            else:
                raise e

    raise RuntimeError("Échec inattendu: sortie de la boucle de retry")

# ==========================================
# 4. CHARGEMENT DES DONNÉES ET REPRISE
# ==========================================
print(f"Chargement des données depuis {FICHIER_ENTREE}...")
df = pd.read_csv(FICHIER_ENTREE)
df.columns = df.columns.str.strip()

# Reprise de la progression existante si le fichier de sortie existe déjà
if os.path.exists(FICHIER_SORTIE):
    print(f"🔄 Fichier existant trouvé ({FICHIER_SORTIE}). Reprise en cours...")
    try:
        df_ex = pd.read_csv(FICHIER_SORTIE, sep='|')
        if "sentiment_final" in df_ex.columns:
            df["sentiment_final"] = df_ex["sentiment_final"]
        else:
            df["sentiment_final"] = pd.Series([None] * len(df))
    except Exception:
        df["sentiment_final"] = pd.Series([None] * len(df))
else:
    df["sentiment_final"] = pd.Series([None] * len(df))

# Trouver les indices restants à traiter
df["sentiment_final"] = pd.to_numeric(df["sentiment_final"], errors='coerce')
indices_a_traiter = df[df["sentiment_final"].isna()].index.tolist()

total_lignes = len(df)
deja_fait = total_lignes - len(indices_a_traiter)
print(f"📊 Progression actuelle : {deja_fait}/{total_lignes} classés.")
print(f"Lignes restantes à classer : {len(indices_a_traiter)}")

# ==========================================
# 5. BOUCLE DE TRAITEMENT PAR LOTS
# ==========================================
delai_entre_lots = 1.2 # Seconde de pause pour respecter le rate limit de Mistral

if indices_a_traiter:
    try:
        for i in tqdm(range(0, len(indices_a_traiter), TAILLE_LOT), desc=f"Classification {MODEL_NAME}"):
            lot_indices = indices_a_traiter[i:i+TAILLE_LOT]
            
            # Formater la requête pour ce lot d'indices
            corps_requete = "Classe les textes suivants (réponds avec l'index de chaque texte) :\n"
            for local_idx, global_idx in enumerate(lot_indices):
                texte = df.loc[global_idx, COLONNE_TEXTE]
                texte_propre = str(texte).replace("\n", " ").replace("\r", " ").strip()
                corps_requete += f"{local_idx}: {texte_propre}\n"

            try:
                donnees_json = appeler_api_avec_retry(corps_requete)
                items = donnees_json.get("resultats", [])
                
                # Mettre à jour le DataFrame principal
                for item in items:
                    local_idx = item.get("index")
                    sentiment = item.get("sentiment")
                    if isinstance(local_idx, int) and 0 <= local_idx < len(lot_indices):
                        global_idx = lot_indices[local_idx]
                        df.at[global_idx, "sentiment_final"] = sentiment if sentiment in [0, 1, 2] else 0

                # Sauvegarde incrémentielle après chaque lot réussi !
                df.to_csv(FICHIER_SORTIE, sep='|', index=False)
                
            except Exception as e:
                print(f"\n❌ Échec persistant pour le lot {lot_indices[0]} à {lot_indices[-1]} : {str(e)[:150]}")
                df.to_csv(FICHIER_SORTIE, sep='|', index=False)

            time.sleep(delai_entre_lots)
            
    except KeyboardInterrupt:
        print("\n🛑 Pause demandée par l'utilisateur. Progression sauvegardée avec succès !")

# Bilan final
manquants = df["sentiment_final"].isna().sum()
print(f"\n✅ Terminé ! {total_lignes - manquants}/{total_lignes} lignes classées.")
if manquants > 0:
    print(f"⚠️ {manquants} lignes encore manquantes. Relance la cellule pour terminer.")
print(f"Fichier sauvegardé ici : {FICHIER_SORTIE}")

Chargement des données depuis D:/XoNet/data/base.csv...
🔄 Fichier existant trouvé (D:/XoNet/data/corpus_final.csv). Reprise en cours...
📊 Progression actuelle : 86976/87126 classés.
Lignes restantes à classer : 150


Classification open-mistral-nemo: 100%|██████████| 1/1 [00:49<00:00, 49.25s/it]


✅ Terminé ! 87126/87126 lignes classées.
Fichier sauvegardé ici : D:/XoNet/data/corpus_final.csv
